In [10]:
import pandas as pd
import numpy as np
import json

In [11]:
PATH_DATA = "data/"

## Official ATIH ICD dictionary

var aut_mco (autorisation MCO)
- 0 Pas de restriction particulière (valeur par défaut). 
- 1 Diagnostic interdit en DP et DR - Autorisé ailleurs 
- 2 Diagnostic interdit en DP et DR - Cause externe de morbidité 
- 3 Diagnostic interdit en DP, DR et DA - Catégories et sous-catégories non vides ou code père interdit 
- 4 Diagnostic interdit en DP – Autorisé ailleurs

In [78]:
df_icd = pd.read_csv(PATH_DATA +"CIM_ATIH_2025/LIBCIM10MULTI.TXT", sep="|",header=None,names=["code","aut_mco","pos","aut_ssr","lib_court","libelle"],encoding="latin-1")
df_icd.code = df_icd.code.str.replace(" ","")
df_icd = df_icd[df_icd.aut_mco!=3]

In [14]:
df_icd_chap20 = pd.read_csv(PATH_DATA +"CIM_ATIH_2025/LIBCIM10MULTI_ch20.TXT", sep="|",header=None,names=["code","aut_mco","pos","aut_ssr","lib_court","libelle"],encoding="latin-1")
df_icd_chap20.code = df_icd_chap20.code.str.replace(" ","")

cat_motif = ["Z"+ str(x).zfill(2) for x in range(0,55)]
cat_facteurs = ["Z"+ str(x).zfill(2) for x in range(55,100)]
cat_sympt = ["R"+ str(x).zfill(2) for x in range(0,100)]

## APHP Dictionnaries (HECTOR)

Détail source Hector :
- A  =  CIM-10 (liste analytique) (ICD-Définition)
- B  = Index CIM-10 (liste alphabétique - référence)
- DR1 =     (sur)
- ED1 =  Dictionaire collégiale endocrinologie (sur)
- GRONES = Groupe NESTOR (sur)
- METABOL = (sur)
- NP1= (sur)
- OP1= (sur)
- ORPHA  = Classification ORHPA Net (sur)
- RH1= (sur)
- SPILFG = Dictionaire Société française de pathologies infectieuses (sur)
- SRLF  = Dictionaire Société française de réanimation (sur)
- T  = Thésam (à fiabiliser)


In [197]:
df_hector_1= pd.read_excel(PATH_DATA + "CIM_APHP_2019/Dictionnaire_Hector_MAJ062019.xlsx",sheet_name="Cim Analytique",names =["libelle","source","code","autre_code"])
df_hector_2= pd.read_excel(PATH_DATA + "CIM_APHP_2019/Dictionnaire_Hector_MAJ062019.xlsx",sheet_name="Cim Alphabétique",names =["libelle","source","code","autre_code"])
df_hector = pd.concat([df_hector_1,df_hector_2],axis =0)
tmp= pd.read_excel(PATH_DATA + "CIM_APHP_2019/Dictionnaire_Hector_MAJ062019.xlsx",sheet_name="Thesam",names =["libelle","source","code","autre_code"])
df_hector = pd.concat([df_hector,tmp],axis =0)
tmp= pd.read_excel(PATH_DATA + "CIM_APHP_2019/Dictionnaire_Hector_MAJ062019.xlsx",sheet_name="Dermatologie",names =["libelle","source","code","autre_code"])
df_hector = pd.concat([df_hector,tmp],axis =0)
tmp= pd.read_excel(PATH_DATA + "CIM_APHP_2019/Dictionnaire_Hector_MAJ062019.xlsx",sheet_name="Endocrinologie",names =["libelle","source","code","autre_code"])
df_hector = pd.concat([df_hector,tmp],axis =0)
tmp= pd.read_excel(PATH_DATA + "CIM_APHP_2019/Dictionnaire_Hector_MAJ062019.xlsx",sheet_name="GRONES",names =["libelle","source","code","autre_code"])
df_hector = pd.concat([df_hector,tmp],axis =0)
tmp= pd.read_excel(PATH_DATA + "CIM_APHP_2019/Dictionnaire_Hector_MAJ062019.xlsx",sheet_name="Troubles métaboliques",names =["libelle","source","code","autre_code"])
df_hector = pd.concat([df_hector,tmp],axis =0)
tmp= pd.read_excel(PATH_DATA + "CIM_APHP_2019/Dictionnaire_Hector_MAJ062019.xlsx",sheet_name="Néphrologie",names =["libelle","source","code","autre_code"])
df_hector = pd.concat([df_hector,tmp],axis =0)
tmp= pd.read_excel(PATH_DATA + "CIM_APHP_2019/Dictionnaire_Hector_MAJ062019.xlsx",sheet_name="Ophtalmo",names =["libelle","source","code","autre_code"])
df_hector = pd.concat([df_hector,tmp],axis =0)
tmp= pd.read_excel(PATH_DATA + "CIM_APHP_2019/Dictionnaire_Hector_MAJ062019.xlsx",sheet_name="Orphanet",names =["libelle","source","code","autre_code"])
df_hector = pd.concat([df_hector,tmp],axis =0)
tmp= pd.read_excel(PATH_DATA + "CIM_APHP_2019/Dictionnaire_Hector_MAJ062019.xlsx",sheet_name="Rhumatologie",names =["libelle","source","code","autre_code"])
df_hector = pd.concat([df_hector,tmp],axis =0)
tmp= pd.read_excel(PATH_DATA + "CIM_APHP_2019/Dictionnaire_Hector_MAJ062019.xlsx",sheet_name="Germes",names =["libelle","source","code","autre_code"])
df_hector = pd.concat([df_hector,tmp],axis =0)
tmp= pd.read_excel(PATH_DATA + "CIM_APHP_2019/Dictionnaire_Hector_MAJ062019.xlsx",sheet_name="SRLF",names =["libelle","source","code","autre_code"])
df_hector = pd.concat([df_hector,tmp],axis =0)

### First list of synonyms 
Exclude :
- B  =  CIM-10 (liste analytique) (ICD-Définition)
- T  = Thésam (à fiabiliser)

ICD-10 index is taken as it is, no transformation.


In [212]:
df_hector_s = df_hector[~ (df_hector.source.isin(["A","T"]) ) ].rename(columns={"libelle":"extract"}).merge(df_icd[["code","libelle"]].rename(columns={"libelle":"definition"}) )[["extract","code","definition"]]
df_hector_s = df_hector_s.assign(label= 1) 

## ORHPANET dictinonary

In [4]:
import requests
from lxml import etree
import pandas as pd

import os
import requests

# 1. Télécharger le XSD depuis l'URL
#xsd_url = "https://www.orphacode.org/data/xsd_jpg/ORPHA_ICD10_mapping_en_2020.xsd"
#response = requests.get(xsd_url)

# Chemin du dossier de destination
#dossier_destination = "data/Orphanet_Nomenclature_Pack_FR_2025/"

# Créer le dossier s'il n'existe pas
#os.makedirs(dossier_destination, exist_ok=True)

# Chemin complet pour le fichier XSD
#chemin_xsd = os.path.join(dossier_destination, "ORPHA_ICD10_mapping_en_2020.xsd")

# 2. Écrire le fichier XSD dans le dossier spécifié
#with open(chemin_xsd, 'wb') as f:
#    f.write(response.content)

In [6]:
path_orphanet = "data/Orphanet_Nomenclature_Pack_FR_2025/"
path_orphanet_xsd = os.path.join(dossier_destination, "ORPHA_ICD10_mapping_en_2020.xsd")

# 3. Charger le schéma XSD
with open(path_orphanet_xsd, 'rb') as f:
    xsd_schema = etree.XMLSchema(file=f)

# 4. Parser le fichier XML avec validation
xml_file = os.path.join(path_orphanet, "ORPHA_ICD10_mapping_fr_2025.xml")

with open(xml_file, 'rb') as f:
    xml_doc = etree.parse(f)
    xsd_schema.assertValid(xml_doc)  # Valide le XML selon le XSD




In [79]:
# Liste pour stocker les données
data = []

# Parcourir chaque élément Disorder
for disorder in xml_doc.xpath('//Disorder'):
    orpha_code = disorder.xpath('OrphaCode/text()')[0]
    name = disorder.xpath('Name/text()')[0]
    name_lang = disorder.xpath('Name/@lang')[0]

    # Extraire les synonymes
    synonyms = disorder.xpath('SynonymList/Synonym/text()')
    synonyms_lang = disorder.xpath('SynonymList/Synonym/@lang')
    synonyms_list = ', '.join([f"{synonym} ({lang})" for synonym, lang in zip(synonyms, synonyms_lang)]) if synonyms else None

    # Extraire les ExternalReferences
    for external_ref in disorder.xpath('ExternalReferenceList/ExternalReference'):
        source = external_ref.xpath('Source/text()')[0]
        reference = external_ref.xpath('Reference/text()')[0]

        # Extraire les relations ICD
        mapping_icd_relation_name = external_ref.xpath('DisorderMappingICDRelation/Name/text()')[0]
        mapping_icd_relation_id = external_ref.xpath('DisorderMappingICDRelation/@id')[0]

        # Ajouter les données à la liste
        data.append({
            'OrphaCode': orpha_code,
            'Name': name,
            'Name_lang': name_lang,
            'SynonymList': synonyms_list,
            'Source': source,
            'Reference': reference,
            'DisorderMappingICDRelation_name': mapping_icd_relation_name,
            'DisorderMappingICDRelation_id': mapping_icd_relation_id,
        })

# Créer le DataFrame
df_orphanet = pd.DataFrame(data)


In [80]:
df_orphanet.columns = df_orphanet.columns.str.lower()
df_orphanet = df_orphanet.assign(reference = df_orphanet.reference.str.replace("\.",""))

C:\Users\3056269\AppData\Local\Temp\ipykernel_28508\1194007567.py:2: FutureWarning: The default value of regex will change from True to False in a future version.
  df_orphanet = df_orphanet.assign(reference = df_orphanet.reference.str.replace("\.",""))


In [81]:
df_orphanet_s = pd.concat([df_orphanet.loc[~ df_orphanet.name.isna(),["reference","name"]].rename(columns={"reference":"code","name":"extract"}),
           df_orphanet.loc[~ df_orphanet.synonymlist.isna(),["reference","synonymlist"]].rename(columns={"reference":"code","synonymlist":"extract"})],axis =0)

In [82]:
df_orphanet_s = df_orphanet_s.merge(df_icd[["code","libelle"]].rename(columns={"libelle" : "definition"}))
df_orphanet_s = df_orphanet_s.assign(label= 1)

In [ ]:
df_prep = pd.concat

# OFS database

In [52]:
df_ofs_master = pd.read_csv(PATH_DATA +"CIM_OFS_SW_2006/MASTER.TXT", sep="¦",encoding="latin-1")
df_ofs_system = pd.read_csv(PATH_DATA +"CIM_OFS_SW_2006/SYSTEM.TXT", sep="¦",encoding="latin-1")
df_ofs_desc = pd.read_csv(PATH_DATA +"CIM_OFS_SW_2006/LIBELLE.TXT", sep="¦",encoding="latin-1", doublequote=False)
df_ofs_include = pd.read_csv(PATH_DATA +"CIM_OFS_SW_2006/INCLUDE.TXT", sep="¦",encoding="latin-1", doublequote=False)
df_ofs_exclude = pd.read_csv(PATH_DATA +"CIM_OFS_SW_2006/EXCLUDE.TXT", sep="¦",encoding="latin-1", doublequote=False)
df_ofs_description = pd.read_csv(PATH_DATA +"CIM_OFS_SW_2006/DESCR.TXT", sep="¦",encoding="latin-1", doublequote=False)

C:\Users\3056269\AppData\Roaming\Python\Python310\site-packages\pandas\util\_decorators.py:311: ParserWarning: Falling back to the 'python' engine because the separator encoded in utf-8 is > 1 char long, and the 'c' engine does not support such separators; you can avoid this warning by specifying engine='python'.
  return func(*args, **kwargs)


### Inclusion

In [171]:
df_icd_inclus =df_ofs_master.loc[df_ofs_master.abbrev.isin(df_icd.code),["SID","abbrev"]].rename(columns={"abbrev":"code"}).\
    merge(df_ofs_description).merge(df_ofs_desc.loc[df_ofs_desc.libelle.notna(),["LID","libelle"]]).rename(columns={"libelle":"extract"})[["SID","code","extract"]]

In [172]:
df_icd_inclus = df_icd_inclus[["code","extract"]].merge(df_icd[["code","libelle"]].rename(columns={"libelle" : "definition"}))
df_icd_inclus = df_icd_inclus.assign(label= 1) 

In [173]:
df_icd_inclus.to_excel("data/inclusions_work.xlsx")

In [207]:
df_icd_inclus = pd.read_excel("data/inclusions_v2.xlsx")

In [208]:
df_icd_inclus = df_icd_inclus.assign(extract = np.where(df_icd_inclus.prefix.notna(),df_icd_inclus.prefix + " " + df_icd_inclus.extract, df_icd_inclus.extract)).drop(columns=["prefix"])


### Exclusion

In [176]:
df_icd_exclus = df_ofs_master.loc[df_ofs_master.abbrev.isin(df_icd.code),["SID","abbrev"]].rename(columns={"abbrev":"code"}).\
    merge(df_ofs_exclude).merge(df_ofs_desc[["LID","libelle"]]).rename(columns={"libelle":"extract"})[["code","extract"]]

In [177]:
df_icd_exclus = df_icd_exclus[["code","extract"]].merge(df_icd[["code","libelle"]].rename(columns={"libelle" : "definition"}))
df_icd_exclus = df_icd_exclus.assign(label= 0) 

In [178]:
df_icd_exclus.to_excel("data/exclusions_work.xlsx")

In [205]:
df_icd_exclus = pd.read_excel("data/exclusions_v2.xlsx")

In [206]:
df_icd_exclus = df_icd_exclus.assign(extract = np.where(df_icd_exclus.prefix.notna(),df_icd_exclus.prefix + " " + df_icd_exclus.extract, df_icd_exclus.extract)).drop(columns=["prefix"])


# Final DF

In [213]:
df = pd.concat([df_hector_s,df_orphanet_s,df_icd_inclus,df_icd_exclus])

In [215]:
df.to_excel("data/dict_syn.xlsx",index=False)